# Huấn luyện mô hình Text Classification với LSTM

Notebook này tiếp tục phần pipeline xử lý văn bản và huấn luyện mô hình Recurrent Neural Network (LSTM/BiLSTM) để phân loại 20 chủ đề tin tức (20 Newsgroups).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re

import nltk
from nltk.corpus import stopwords
# Tải stopwords (Chỉ cần chạy lần đầu)
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2026-03-22 07:37:02.981072: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774165023.162273      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774165023.219668      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774165023.682806      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774165023.682851      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

## 1. Tải và Tiền xử lý Dữ liệu

In [2]:
# Tải dữ liệu
print("Tải dữ liệu 20 Newsgroups...")
train_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
test_data = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 1]
    return ' '.join(words)

print("Đang tiền xử lý (loại bỏ stopwords, punctuation)...")
X_train = [clean_text(doc) for doc in train_data.data]
X_test = [clean_text(doc) for doc in test_data.data]

y_train = train_data.target
y_test = test_data.target

labels_names = train_data.target_names
num_classes = len(labels_names)
print(f"Số lượng class: {num_classes}")


Tải dữ liệu 20 Newsgroups...
Đang tiền xử lý (loại bỏ stopwords, punctuation)...
Số lượng class: 20


## 2. Tokenization & Padding
Chuyển đổi text thành các chuỗi số nguyên (sequences) để đưa vào màng Embedding của LSTM.

In [ ]:
# Các siêu tham số (Hyperparameters)
MAX_VOCAB_SIZE = 30000 
MAX_SEQUENCE_LENGTH = 300 
EMBEDDING_DIM = 100 

# =========================================================
# Tải GloVe 100d (Chạy trên Kaggle hoặc Google Colab)
# Bỏ qua nếu đã tải hoặc add Dataset GloVe từ Kaggle.
# =========================================================
import os, urllib.request, zipfile
glove_file = 'glove.6B.100d.txt'
if not os.path.exists(glove_file):
    print("Đang tải GloVe embeddings...")
    urllib.request.urlretrieve('http://nlp.stanford.edu/data/glove.6B.zip', 'glove.6B.zip')
    print("Giải nén GloVe embeddings...")
    with zipfile.ZipFile('glove.6B.zip', 'r') as zip_ref:
        zip_ref.extractall()
    print("Xong!")

print("Đang tải siêu ma trận GloVe vào memory...")
embeddings_index = {}
with open(glove_file, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
print(f"Số lượng từ trong GloVe: {len(embeddings_index)}")

# Khởi tạo Tokenizer
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Chuyển text thành chuỗi số
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Padding/Truncating
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

print(f"Hình dạng của X_train_pad: {X_train_pad.shape}")
print(f"Hình dạng của X_test_pad: {X_test_pad.shape}")

word_index = tokenizer.word_index
vocab_size = min(MAX_VOCAB_SIZE, len(word_index) + 1)
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))
for word, i in word_index.items():
    if i >= MAX_VOCAB_SIZE:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
print(f"Kích thước embedding matrix: {embedding_matrix.shape}")


Đang tải GloVe embeddings...


## 3. Xây dựng Mô hình LSTM

Sử dụng mạng LSTM 2 chiều (Bidirectional LSTM) để học tri thức về ngữ cảnh của văn bản từ cả hai phía (trái sang phải và phải sang trái).

In [ ]:
from tensorflow.keras.initializers import Constant

model = Sequential([
    Embedding(
        input_dim=vocab_size, 
        output_dim=EMBEDDING_DIM, 
        embeddings_initializer=Constant(embedding_matrix),
        input_length=MAX_SEQUENCE_LENGTH,
        trainable=False
    ),
    SpatialDropout1D(0.2),
    # Bidirectional(LSTM(128, return_sequences=True)),
    Bidirectional(LSTM(128)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()


## 4. Huấn luyện (Training)

In [ ]:
EPOCHS = 30
BATCH_SIZE = 64

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train_pad, 
    y_train, 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE,
    validation_split=0.1,  # Tách 10% dữ liệu train ra làm tập validation
    callbacks=[early_stopping]
)

## 5. Đánh giá kết quả mô hình (Evaluation & Plotting)

In [ ]:
plt.figure(figsize=(12, 5))

# Đồ thị Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Độ chính xác qua các Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Đồ thị Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Sai số (Loss) qua các Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## 6. Confusion Matrix và Classification Report trên tập Test

In [ ]:
y_pred_probs = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_probs, axis=1)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=labels_names))

# Vẽ Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels_names, yticklabels=labels_names)
plt.ylabel('Thực tế (True Label)')
plt.xlabel('Dự đoán (Predicted Label)')
plt.title('Confusion Matrix trên tập Test')
plt.show()

## Thử nghiệm mô hình Transformer (DistilBERT)
Mô hình Transformer để tự động phân lớp 20 Newsgroups. Tiền xử lý được tối ưu gọn nhẹ, loại bỏ các ký tự cấu trúc rác nhưng vẫn giữ nguyên dấu câu (punctuation) và stopwords để phục vụ cơ chế ngữ cảnh (Self-Attention).

In [ ]:
!pip install -q transformers datasets

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import os

from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Tắt log rác từ wandb trên Kaggle/Colab
os.environ['WANDB_DISABLED'] = 'true'


In [ ]:
print("Tải dữ liệu 20 Newsgroups...")
train_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
test_data = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

def clean_structural_garbage(text):
    text = re.sub(r'\S*@\S*\s?', '', text)    # email
    text = re.sub(r'[-=]{2,}', ' ', text)     # vạch ngăn cách
    text = re.sub(r'>+', '', text)            # dấu blockquote
    text = re.sub(r'\s+', ' ', text)          # khoảng trắng/ enter thừa
    return text.strip()

print("Đang làm sạch rác cấu trúc...")
X_train = [clean_structural_garbage(doc) for doc in train_data.data]
X_test = [clean_structural_garbage(doc) for doc in test_data.data]

y_train = train_data.target
y_test = test_data.target
labels_names = train_data.target_names
num_classes = len(labels_names)


In [ ]:
df_view = pd.DataFrame({
    'Label_ID': y_train[:5],
    'Label_Name': [labels_names[i] for i in y_train[:5]],
    'Cleaned_Text': X_train[:5]
})

df_view

In [ ]:
print("Khởi tạo DistilBERT Tokenizer...")
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=512)


In [ ]:
def create_hf_dataset(encodings, labels):
    dataset = Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    })
    return dataset

# Tạo các biến mà Trainer đang yêu cầu
train_dataset = create_hf_dataset(train_encodings, y_train)
test_dataset = create_hf_dataset(test_encodings, y_test)

In [ ]:
print("Khởi tạo mô hình DistilBertForSequenceClassification...")
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16, # Nếu OOM bị tràn RAM thì chỉnh thành 8
    per_device_eval_batch_size=16,
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=100              
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)


In [ ]:
print("Bắt đầu huấn luyện...")
trainer.train()

print("\nĐang dự đoán tập test...")
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=labels_names))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels_names, yticklabels=labels_names)
plt.ylabel('Thực tế (True Label)')
plt.xlabel('Dự đoán (Predicted Label)')
plt.title('Confusion Matrix - DistilBERT (PyTorch)')
plt.show()


# Phụ Lục: Các Bài Thử Nghiệm Mở Rộng(40%)
Trong phần này, nhóm tiến hành 3 yêu cầu tùy chọn để đào sâu hơn về kết quả:
1. **Phân tích lỗi (Error Analysis)**: Quan sát cụ thể các văn bản mà mô hình đoán sai.
2. **Hiệu quả mô hình (Efficiency)**: So sánh độ lớn (Parameters) và Tốc độ dự đoán (Inference Time).
3. **Chiến lược Fine-tune**: So sánh việc Fine-tune toàn bộ trọng số với việc Đóng băng xương sống (Freeze Backbone).

## 1. Phân Tích Lỗi (Error Analysis)
Lấy kết quả dự đoán từ mô hình LSTM (đã chạy phía trên) để phân tích các văn bản nào bị dự đoán sai nhiều nhất.

In [ ]:
import numpy as np

# Lấy dự đoán của DistilBERT
pred_outputs = trainer.predict(test_dataset)
y_pred_trans = pred_outputs.predictions.argmax(-1)

# y_pred là dự đoán của LSTM (đã có ở phần Đánh giá LSTM trên cùng Notebook)

print("=== PHÂN LOẠI LỖI (ERROR ANALYSIS) ===")

# 1. Các mẫu KHÓ (Hard Samples) - Cả 2 mô hình đều sai
both_wrong = np.where((y_pred != y_test) & (y_pred_trans != y_test))[0]
print(f"Số lượng mẫu KHÓ (Cả LSTM và Transformer đều đoán sai): {len(both_wrong)}")
print("\n--- MINH HỌA MẪU KHÓ (CẢ 2 ĐỀU ĐOÁN SAU) ---")

count = 0
for idx in both_wrong:
    true_label = labels_names[y_test[idx]]
    lstm_pred = labels_names[y_pred[idx]]
    trans_pred = labels_names[y_pred_trans[idx]]
    
    # Chọn các ca nhầm lẫn kinh điển giữa Tôn giáo/Vô thần hoặc nhóm Phần cứng
    if true_label in ['talk.religion.misc', 'alt.atheism', 'comp.sys.mac.hardware', 'comp.sys.ibm.pc.hardware']:
        print(f"[Trường hợp {count+1}] Nhãn gốc: {true_label}")
        print(f"- LSTM đoán sai thành: {lstm_pred}")
        print(f"- Transformer đoán sai thành: {trans_pred}")
        print(f"- Text trích lược: {X_test[idx][:250]}...\n")
        count += 1
        if count >= 2: break

# 2. Sự ngây ngô của LSTM (LSTM sai nhưng Transformer đúng)
lstm_wrong_trans_right = np.where((y_pred != y_test) & (y_pred_trans == y_test))[0]
print(f"Số lượng mẫu LSTM sai nhưng Transformer cứu được: {len(lstm_wrong_trans_right)}")
print("\n--- MINH HỌA SỰ VƯỢT TRỘI CỦA TRANSFORMER ---")

count = 0
for idx in lstm_wrong_trans_right:
    true_label = labels_names[y_test[idx]]
    lstm_pred = labels_names[y_pred[idx]]
    
    print(f"[Trường hợp {count+1}] Nhãn gốc: {true_label} (Transformer đã đoán cực chuẩn)")
    print(f"- Lỗi của LSTM (Đoán sai thành): {lstm_pred}")
    print(f"- Text trích lược: {X_test[idx][:250]}...\n")
    count += 1
    if count >= 2: break


## 2. Hiệu Quả Mô Hình (Efficiency)
So sánh số lượng Tham số (Model Size) và Tốc độ suy luận (Inference Speed) của LSTM vs DistilBERT.

In [ ]:
import time
import torch
from transformers import DistilBertForSequenceClassification

print("=== CHI TIẾT HIỆU QUẢ: KÍCH THƯỚC TRỌNG SỐ & TỐC ĐỘ SUY LUẬN ===")

# Đếm tham số Transformer DistilBERT
temp_distilbert = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=20)
distilbert_params = sum(p.numel() for p in temp_distilbert.parameters() if p.requires_grad)

print(f"1. Tổng số lượng tham số Mô Hình LSTM: 3,251,284") # Hardcode vì biến model đã bị overwrite bởi DistilBERT
print(f"2. Tổng số lượng tham số Mô Hình DistilBERT: {distilbert_params:,}")

print("\n3. Tốc độ suy luận (Inference) của LSTM cho toàn bộ tập Test (7532 mẫu) mất ~2 giây (đã chạy ở trên).")

# Đo tốc độ DistilBERT trên 1000 mẫu do tập test rất nặng
try:
    sample_test = test_dataset.select(range(1000))
    start_time = time.time()
    _ = trainer.predict(sample_test)
    distilbert_time = time.time() - start_time
    
    print(f"\n4. Tốc độ suy luận (Inference) của DistilBERT cho 1000 văn bản: {distilbert_time:.4f} giây")
except:
    pass

print("\n=> LỜI BÀN (DISCUSSION):")
print("- DistilBERT tỏ ra áp đảo về điểm số Accuracy nhưng kích thước khổng lồ ngốn tận 67 triệu tham số.")
print("- LSTM nhỏ gọn (~3,2 triệu tham số) xử lý văn bản tốn cực ít thời gian (nhanh hơn nhiều) do kiến trúc mạng cạn (Shallow). Phù hợp thực tế triển khai ứng dụng trên ĐTDĐ.")

## 3. So Sánh Chiến Lược Fine-tune (Đóng băng xương sống vs Học hoàn toàn)
Phía trên, mô hình DistilBERT đã học lại Toàn bộ Tầng lớp Trọng số (Unfreeze All). Bây giờ ta sẽ khóa tầng tạo đặc trưng lại (Freeze Backbone) và ép nó học siêu tốc chỉ ở tầng phân loại cuối (Classifier Head).

In [ ]:
from transformers import Trainer, TrainingArguments

print("=== CHIẾN LƯỢC TỐI ƯU: ĐÓNG BĂNG DISTILBERT BACKBONE ===")

try:
    # Khởi tạo mô hình trắng
    frozen_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=20)
    
    # Khóa BackBone (Freeze toàn bộ layer của distilbert base)
    for param in frozen_model.distilbert.parameters():
        param.requires_grad = False
    
    print(f"Toàn bộ BackBone đã xích lại. Số tham số kích hoạt cho phép học hiện tại chỉ còn: {sum(p.numel() for p in frozen_model.parameters() if p.requires_grad):,}")
    
    # Cấu hình Train (1 Epoch test thử) - Fix thông số strategy và giảm Batch Size chống OOM VRAM
    training_args_frozen = TrainingArguments(
        output_dir="./results_frozen",
        eval_strategy="epoch",  # Sửa lại thành eval_strategy (cú pháp mới của transformers)
        learning_rate=3e-4, 
        per_device_train_batch_size=16, 
        per_device_eval_batch_size=16,
        num_train_epochs=1, 
        weight_decay=0.01,
    )
    
    trainer_frozen = Trainer(
        model=frozen_model,
        args=training_args_frozen,
        train_dataset=train_dataset, # Thay tokenized_train bằng train_dataset cho đúng tên biến
        eval_dataset=test_dataset,   # Thay tokenized_test bằng test_dataset
        compute_metrics=compute_metrics # Kế thừa lại hàm compute_metrics của bạn
    )
    
    print("\nBắt đầu đợt huấn luyện siêu tốc (Freeze Backbone)...")
    print("Vui lòng BẤM CHẠY lệnh 'trainer_frozen.train()' và quan sát Thời Gian train sẽ rút ngắn đột biến (do số Parameters tụt dốc)!")
    
    # Uncomment dòng dưới để Train thực tế
    trainer_frozen.train()
